In [1]:
import numpy as np
from tqdm import tqdm
import pandas as pd
import json
import time
import re
import requests
from functools import wraps
import subprocess
import utils # Custom python utility functions

# Assume data is already converted to javabin format and placed in appropriate directory.
# mkdir -p batches_1M
# java -cp solr-cuvs-benchmarks-1.0-SNAPSHOT-jar-with-dependencies.jar com.searchscale.benchmarks.Indexer \ 
#     data_file=wiki_dump_5Mx2048D.csv.gz output_file=batches/wiki batch_size=50000 docs_count=1000000 legacy=true

data_dir = 'batches_50k'
use_hw = 'cpu'
jvm_mem = '4G'
solr_url = 'http://localhost:8983/solr/test/select'

# Configure Solr

In [2]:
# Templates for parameter sweeps
if use_hw == 'cpu':
    model_name = 'hnsw'
    model_params = {
        'dim': 2048,
        'hnswMaxConnections': 32,
        'hnswBeamWidth': 512
    }
elif use_hw == 'gpu':
    model_name = 'cuvs'
    model_params = {
        'dim': 2048,
        'graphDegree': 32,
        'intGraphDegree': 64,
        'cuvsWriterThreads': 8
    }
else:
    raise ValueError('Unknown use_hw value. Choose "cpu" or "gpu".')
    
# Generate solr xml config files
utils.generate_config_xml(model_name, model_params)

# Generate Solr bash scripts
utils.generate_solr_bash_scripts(data_dir, use_hw, jvm_mem)
    
# Start Solr and reset database
subprocess.run("chmod +x *.sh", shell=True, executable="/bin/bash")
print()
subprocess.run("sh ./start_solr_mod.sh", shell=True, executable="/bin/bash")
print()

Generating hnsw schema.xml and solrconfig.xml files.
Completed writing all XML files.
Sucessfully written ./start_solr_mod.sh and ./upload_all_files_mod.sh.

Waiting up to 180 seconds to see Solr running on port 8983 [/]  
Started Solr server on port 8983 (pid=9971). Happy searching!



  adding: schema.xml (deflated 60%)
  adding: solrconfig.xml (deflated 52%)
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2031  100    60  100  1971     30    998  0:00:02  0:00:01  0:00:01  1028
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

{
  "responseHeader":{
    "status":0,
    "QTime":160
  }
}{
  "responseHeader":{
    "status":0,
    "QTime":1198
  },
  "success":{
    "localhost:8983_solr":{
      "responseHeader":{
        "status":0,
        "QTime":810
      },
      "core":"test_shard1_replica_n1"
    }
  }
}


100   226  100   226    0     0    188      0  0:00:01  0:00:01 --:--:--   188


In [3]:
# Start javabin upload and indexing pipeline
start_time = time.perf_counter()
subprocess.run("sh ./upload_all_files_mod.sh", shell=True, executable="/bin/bash")
indexing_time = time.perf_counter() - start_time

print('Indexing time:', f'{indexing_time:.2f} seconds')

Uploading batches_50k/wiki.0...
{
  "responseHeader":{
    "rf":1,
    "status":0,
    "QTime":129093
  }
}All files in the directory uploaded.
Indexing time: 129.46 seconds


# Vector Search

In [4]:
# Specify search parameters:
topK = 10
return_limit = topK
num_queries = 1024

if use_hw == 'cpu':
    # Build params: hnswBeamWidth=efConstruction, hnswMaxConnections=M.
    # No efSearch parameter. Use default value.
    query_prefix = f'{{!knn f=article_vector topK={topK}}}'
elif use_hw == 'gpu':
    cagraITopK = 10
    cagraSearchWidth = 32
    query_prefix = f'{{!cuvs f=article_vector cagraITopK={cagraITopK} cagraSearchWidth={cagraSearchWidth} topK={topK}}}'
else:
    raise ValueError('Unknown use_hw value. Choose "cpu" or "gpu".')


# Load query vectors
ids, query_vector_strs = utils.load_query_vectors(f'{data_dir}/wiki.0', num_queries)

SLF4J(W): No SLF4J providers were found.
SLF4J(W): Defaulting to no-operation (NOP) logger implementation
SLF4J(W): See https://www.slf4j.org/codes.html#noProviders for further details.


Successfully loaded query vectors.


In [5]:
def time_it(func: any):
    """returns result and elapsed time"""
    @wraps(func)
    def inner(*args, **kwargs):
        pref = time.perf_counter()
        result = func(*args, **kwargs)
        delta = time.perf_counter() - pref
        return result, delta
    return(inner)

@time_it
def run_single_query(query_prefix, vector_str, return_limit=10):
    """
    Function for submitting individual queries to Solr.
    """
    
    query_obj = {
      "query": {
        "lucene": {
          "df": "name",
          "query": query_prefix + vector_str
        }
      },
      "fields": "id",
      "limit": return_limit
    }

    response = requests.post(solr_url, json = query_obj)

    # Convert response text to dict:
    response = json.loads(response.text)

    # Initialize outputs
    out = {}

    # Validate responses
    if response['responseHeader']['status'] == 0:
        out['QTime'] = response['responseHeader']['QTime']
    else:
        raise ValueError('Query did not complete successfully.')

    # Accumulate doc_ids
    out['doc_ids'] = [d['id'] for d in response['response']['docs']]
    return(out)

def run_all_queries(query_prefix, query_vector_strs, batch_size=1):
    print(f'batch_size={batch_size}, num_queries={num_queries}')
    print(model_params)
    start_time = time.perf_counter()
    query_results = [run_single_query(query_prefix, vector_str) for vector_str in tqdm(query_vector_strs, desc='Run progress')]
    elapsed_time = time.perf_counter() - start_time

    # Store matches
    topK_ids = [r[0] for r in query_results]
    topK_ids = [[int(ii) for ii in topK_ids[jj]['doc_ids']] for jj in range(len(query_results))]
    
    # Assemble run times
    timing_store = [r[1] for r in query_results]
    run_times = pd.Series(timing_store)
    P99 = run_times.quantile(.99)
    
    print('Wall time:', f'{elapsed_time:.2f} seconds')
    print(f'QPS={num_queries/elapsed_time:.2f}, P99={P99*1000:0.2f} ms')
    print()
    return(topK_ids)

topK_ids = run_all_queries(query_prefix, query_vector_strs)

batch_size=1, num_queries=1024
{'dim': 2048, 'hnswMaxConnections': 32, 'hnswBeamWidth': 512}


Run progress: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1024/1024 [00:05<00:00, 191.76it/s]

Wall time: 5.35 seconds
QPS=191.53, P99=9.67 ms



In [6]:
topK_ids[0]

[39, 42986, 108115, 16343, 68915, 7517, 110736, 74969, 110319, 67337]